In [ ]:
from pathlib import Path
import numpy as np
from skimage import io
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift, binary_dilation, gaussian_filter
import matplotlib.pyplot as plt

# ------------------------------
# 1. Load masks
# ------------------------------
# Assume masks are TIFFs: (Z,Y,X)
file_list = sorted(list(Path("arabidopsis/timepoints").glob("*.tif")))
masks = [io.imread(str(f)) for f in file_list]
print("Loaded masks:", [m.shape for m in masks])

# Find the max shape along each axis
max_shape = np.max([m.shape for m in masks], axis=0)

# Pad masks to max shape
def pad_to_shape(mask, target_shape):
    pad_width = [(0, ts - s) for s, ts in zip(mask.shape, target_shape)]
    return np.pad(mask, pad_width, mode='constant', constant_values=0)

masks_padded = [pad_to_shape(m, max_shape) for m in masks]

# Use masks_padded for alignment
ref_mask = masks_padded[0]
shifts = [(0,0,0)]
for i, mask in enumerate(masks_padded[1:], start=1):
    shift_est, _, _ = phase_cross_correlation(ref_mask, mask, upsample_factor=10)
    shifts.append(shift_est)
    print(f"Timepoint {i+1}: estimated shift {shift_est}")
masks = masks_padded

# Convert to float32 and binary (presence/absence)
masks = [(m > 0).astype(np.float32) for m in masks]

# ------------------------------
# 2. Compute shifts to align masks
# ------------------------------
ref_mask = masks[0]
shifts = [(0, 0, 0)]  # first timepoint is reference

for i, mask in enumerate(masks[1:], start=1):
    shift_est, _, _ = phase_cross_correlation(ref_mask, mask, upsample_factor=10)
    shifts.append(shift_est)
    print(f"Timepoint {i+1}: estimated shift {shift_est}")

# ------------------------------
# 3. Apply shifts and build probability map
# ------------------------------
aligned_masks = []
for mask, sh in zip(masks, shifts):
    # Apply shift (nearest-neighbor to keep binary)
    aligned = shift(mask, shift=sh, order=0, mode="nearest")
    # Optionally dilate in XY to thicken walls
    aligned = binary_dilation(aligned, structure=np.ones((1,3,3))).astype(np.float32)
    # Gaussian smoothing (anisotropic to preserve Z thin walls)
    aligned = gaussian_filter(aligned, sigma=(0.5,1.5,1.5))
    aligned_masks.append(aligned)

aligned_stack = np.stack(aligned_masks, axis=0)  # (time, Z, Y, X)
prob_map = aligned_stack.mean(axis=0)  # probability map in [0,1]

# ------------------------------
# 4. Make corrected timepoints
# ------------------------------
threshold = 0.2  # probability threshold to accept voxel
corrected_masks = []
for aligned, sh in zip(aligned_masks, shifts):
    corrected = (aligned > threshold).astype(np.uint8)
    # Shift back to original coordinates
    corrected_unshift = shift(corrected, shift=-sh, order=0, mode="nearest")
    corrected_masks.append(corrected_unshift)
    # Save TIFF
    io.imsave(f"corrected_timepoint_{len(corrected_masks):02d}.tif",
              (corrected_unshift * 255).astype(np.uint8))

# ------------------------------
# 5. Visualization
# ------------------------------
# Probability map: middle slice
z_mid = prob_map.shape[0] // 2
plt.figure(figsize=(6,6))
plt.imshow(prob_map[z_mid], cmap='gray', vmin=0, vmax=1)
plt.title("Middle slice probability map")
plt.axis('off')
plt.show()

# Probability map: max projection along Z
plt.figure(figsize=(6,6))
plt.imshow(prob_map.max(axis=0), cmap='gray', vmin=0, vmax=1)
plt.title("Max projection probability map")
plt.axis('off')
plt.show()


Loaded masks: [(157, 526, 524), (173, 526, 524), (168, 526, 524), (170, 526, 524)]
Timepoint 2: estimated shift [ -1.6  33.  -11.9]
Timepoint 3: estimated shift [6.7 7.5 0.9]
Timepoint 4: estimated shift [ 2.3 15.5  3.7]
Timepoint 2: estimated shift [ -82. -104.   86.]
Timepoint 3: estimated shift [ 14.9 130.   31. ]
Timepoint 4: estimated shift [ -66.   36. -201.]


TypeError: bad operand type for unary -: 'tuple'

_close_app() missing 1 required positional argument: 'window'
Traceback (most recent call last):
  File "/Users/tomkinsm/miniconda3/lib/python3.12/site-packages/in_n_out/_store.py", line 804, in _exec
    result = func(**bound.arguments)
             ^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _close_app() missing 1 required positional argument: 'window'
Do not have argument for window: using providers [<function _provide_window at 0x37fa98f40>]
Traceback (most recent call last):
  File "/Users/tomkinsm/miniconda3/lib/python3.12/site-packages/in_n_out/_store.py", line 804, in _exec
    result = func(**bound.arguments)
             ^^^^^^^^^^^^^^^^^^^^^^^
TypeError: _close_app() missing 1 required positional argument: 'window'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/tomkinsm/miniconda3/lib/python3.12/site-packages/app_model/backends/qt/_qaction.py", line 59, in _on_triggered
    self._app.commands.execute_command(se

: 

In [17]:
import napari

stack = io.imread("arabidopsis/timepoints/Sample2_myrYFP_6_Dark_t0_2025_08_06__13_41_44_Out.tif")
viewer = napari.view_image(stack, name="corrected mask", colormap='gray')
napari.run()


/var/folders/t5/q1tt12hd2sg29kw894zcfbtm000b_5/T/ipykernel_97044/950770555.py:4: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(stack, name="corrected mask", colormap='gray')
